# 09 — The label pathology (the headline finding)

**The shipped NNN training labels at T9 ≳ 5 carry the MESA gh-575 /
Appendix-B inverse-rate bug.** Not one sampled hot state is near NSE
(frac > 1 dex = **1.000** in every T9 bin, both networks); the per-bin
*medians* rise 6.9 → 11.5 dex (mesa_80) and 6.7 → 13.0 dex (mesa_151)
across bins [5.0,5.5)…[7.0,7.94). The labels are parked at *displaced
pseudo-equilibria* — si30-class at high ρ, c12/o16-class at low ρ — and on
the witness states this composition is frozen across dt decades. The
labels' Yₑ evolution itself is wrong there (0.4713 vs 0.4619 at the witness
state).

The attribution is **dispositive**, and this is what makes it a finding
rather than a suspicion:

| leg | result |
|---|---|
| stock r23.05.1 bbq at label conditions | reproduces the labels to **0.01–0.03 dex**, displaced attractor included |
| MESA 24.08.1 (gh-575 fixed) | relaxes to **NSE** |
| our pf-true integrator + independent Saha solver | agree with 24.08.1 |

Controlling channels: the chapter-8 1→3 reverses (c12→3α class) found
"beyond the paper" in Step 4. This supersedes the earlier Step-4 statement
that the labels used the authors' FIXED MESA. The Step-5 trajectory "stall"
(notebook 07) is arrival at these attractors. The novelty sweep found the
finding unclaimed in the literature (queries returned zero third-party
discussion) — candidate standalone publication. Note the framing the sweep
insists on: the *existence* of the bug is the NNN authors' own disclosure
(their App. B); what is new here is the measured CONSEQUENCE for the
shipped training labels.

Exploratory only — citable values are the RESULTS.md 2026-07-11 rows.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np

import nbsupport as nbs
from gnn_nucleo.data.labels import load_step_frame
from gnn_nucleo.data.schema import DT_LABELS
from gnn_nucleo.data.subsample import load_subsample_ids
from gnn_nucleo.graph import load_isotope_table
from gnn_nucleo.qse import build_inputs, solve_nse

nbs.style()
QUICK = nbs.QUICK
NET = "mesa_80"
N_PER_BIN = 30 if QUICK else 120
T9_BINS = [(5.0, 5.5), (5.5, 6.0), (6.0, 6.5), (6.5, 7.0), (7.0, 7.94)]
#: each label CSV is a multi-GB scan (~13 s), so QUICK samples 3 dts spanning
#: the same eight decades rather than all nine.
DT_SHOW = ["1e-6", "1e0", "1e2"] if QUICK else list(DT_LABELS)

In [ ]:
nbs.provenance_header(
    "09",
    "The label pathology — bug-displaced training labels at T9 ≳ 5",
    nbs.status_of([8]),
    results_rows=[
        "2026-07-11: label-NSE census (120 states/bin, T9 > 5) — frac > 1 dex = 1.000 in every bin, both nets (NO sampled high-T9 label endpoint is near NSE); per-bin MEDIAN rises 6.9 → 11.5 dex (mesa_80) / 6.7 → 13.0 dex (mesa_151)",
        "2026-07-11: attribution — stock r23.05.1 bbq reproduces the shipped labels to 0.01/0.03 dex INCLUDING the bug-displaced pseudo-equilibrium; MESA 24.08.1 relaxes to NSE",
        "2026-07-11: labels' Yₑ evolution wrong at T9 ≳ 5 — 0.4713 (label) vs 0.4619 (24.08.1) at the witness state",
        "2026-07-09: Appendix-B — stock r23.05.1 HAS the gh-575 bug (|Δlog10| 10.0–11.1 at |ΔN|=1, 20.6–22.7 at |ΔN|=2); BEYOND the paper: the chapter-8 1→3 reverses (c12→3α class) low by 9.5–11.3 dex — the controlling channels here",
    ],
    data=[
        "data/zenodo/.../training_sets/mesa_80/mesa_80_1e2_sec.csv (labels; READ-ONLY)",
        "data/bbq_reruns/mesa_80/diag_state80_{stock,24081} (three-way witness runs)",
    ],
    scripts=[
        "scripts/step6_label_nse_census.py",
        "scripts/appendixb_check.py",
        "docs/novelty/2026-07-11-step6-phase-boundary.md",
    ],
)

## Load labels + the independent NSE solver

The comparison leg is our own Saha/NSE solver (`gnn_nucleo.qse`) — an
implementation entirely independent of MESA, which is what lets it arbitrate.
Distance metric ported from `scripts/step6_label_nse_census.py`:
`max |Δlog10 X|` over species with X > 1e-6 in either composition.

In [ ]:
tab = load_isotope_table(NET)
names = list(tab.names)
A = tab.A.astype(float)
Z = tab.Z.astype(float)
inputs = build_inputs(NET)
ids = load_subsample_ids(NET)[:12000]

cols = ["logT", "logRho"] + [f"final_{n}" for n in names]
df = load_step_frame(NET, "1e2", columns=cols, state_ids=ids)
t9 = 10.0 ** df["logT"].to_numpy() / 1e9
rho = 10.0 ** df["logRho"].to_numpy()
Xf = df[[f"final_{n}" for n in names]].to_numpy()
print(f"{NET}: {len(df)} subsample states at dt = 1e2 s (the longest label step — "
      "if anything has relaxed, it has relaxed by here)")


def nse_distance(X, t9_, rho_):
    """max |Δlog10 X| between a composition and true NSE (ported from
    scripts/step6_label_nse_census.py:nse_distance)."""
    ye = float(X @ (Z / A))
    nse = solve_nse(inputs, t9_ * 1e9, rho_, ye)
    if not nse.converged:
        return np.nan, None
    m = (X > 1e-6) | (nse.X > 1e-6)
    d = np.abs(np.log10(np.maximum(X[m], 1e-15)) - np.log10(np.maximum(nse.X[m], 1e-15))).max()
    return float(d), nse

## Figure 1 — how far the labels sit from NSE, by temperature

At T9 ≳ 5 with a dt of 100 s, silicon burning **must** be at NSE. The labels
are not.

In [ ]:
dist = {}
for lo, hi in T9_BINS:
    sel = np.nonzero((t9 >= lo) & (t9 < hi))[0][:N_PER_BIN]
    ds = np.array([d for d in (nse_distance(Xf[k], t9[k], rho[k])[0] for k in sel)
                   if np.isfinite(d)])
    dist[(lo, hi)] = ds
    print(f"  T9 [{lo},{hi}): n {len(ds):>4}  median {np.median(ds):6.2f} dex  "
          f"frac>1dex {(ds > 1).mean():.3f}  frac>3dex {(ds > 3).mean():.3f}")

fig, ax = plt.subplots(figsize=(9.5, 4.8))
labels = [f"[{lo},{hi})" for lo, hi in T9_BINS]
ax.boxplot([dist[b] for b in T9_BINS], tick_labels=labels, showfliers=True,
           flierprops=dict(marker=".", ms=3))
ax.axhline(1.0, color="#D55E00", ls="--", lw=1.2, label="1 dex from NSE")
ax.set_xlabel("T₉ bin")
ax.set_ylabel("max |Δlog₁₀ X| from true NSE\n(species with X > 1e-6)")
ax.set_title(f"{NET} labels at dt = 1e2 s vs an independent Saha/NSE solve")
ax.legend(fontsize=8)
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "Every hot bin sits many dex from NSE — the measured per-bin MEDIANS rise 6.9→11.5 dex "
    "(mesa_80) / 6.7→13.0 dex (mesa_151), with frac>1dex = 1.000 in every bin, i.e. NOT ONE "
    "sampled hot state is at NSE. This is "
    "the labels the published NNN was trained on, and the labels our Phase-1 training would "
    "inherit. Quick-look subset here; the census is the cited row.",
    results=["RESULTS.md 2026-07-11 label-NSE census rows (scripts/step6_label_nse_census.py, commit 3479512)"],
    scripts=["scripts/step6_label_nse_census.py"],
)

## Figure 2 — the displaced attractors: what the labels say vs what NSE says

Witness state 80 (T9 7.08, ρ 3.0e8 — the si-group attractor class). At this
temperature NSE is an iron-peak composition; the label is silicon-group.
This is not a small error, and it is not noise.

In [ ]:
WITNESS_ID = 80  # scripts/step6_label_nse_census.py WITNESS: si-group attractor (T9 7.08, rho 3.0e8)
pos = int(np.nonzero(ids == WITNESS_ID)[0][0]) if (ids == WITNESS_ID).any() else 0
d_w, nse_w = nse_distance(Xf[pos], t9[pos], rho[pos])
print(f"witness state_id {ids[pos]}: T9 = {t9[pos]:.2f}, rho = {rho[pos]:.2e}, "
      f"Yₑ = {Xf[pos] @ (Z / A):.4f} — distance from NSE {d_w:.2f} dex")

Xl = Xf[pos]
Xn = nse_w.X
show = np.argsort(-np.maximum(Xl, Xn))[:12]
fig, ax = plt.subplots(figsize=(11, 4.6))
x = np.arange(len(show))
ax.bar(x - 0.2, np.maximum(Xl[show], 1e-16), 0.4, label="shipped LABEL (dt = 1e2 s)", color="#D55E00")
ax.bar(x + 0.2, np.maximum(Xn[show], 1e-16), 0.4, label="true NSE (independent Saha solve)",
       color="#009E73")
ax.set_yscale("log")
ax.set_ylim(1e-12, 2)
ax.set_xticks(x, [names[i] for i in show], rotation=35, ha="right")
ax.set_ylabel("mass fraction X")
ax.set_title(f"Witness state {ids[pos]} (T₉ = {t9[pos]:.2f}, ρ = {rho[pos]:.1e}) — "
             f"label vs NSE, {d_w:.1f} dex apart")
ax.legend(fontsize=8)
nbs.caption(
    fig,
    "The label is parked on a si30-class pseudo-equilibrium while true NSE at these conditions is "
    "iron-peak. High-ρ states show the si30 class, low-ρ states a c12/o16 class. The mechanism is "
    "MESA's bugged multi-body inverse rates (notebook 04): with reverse flows wrong by 10+ dex, "
    "the network converges to a fixed point that isn't equilibrium.",
    results=[
        "RESULTS.md 2026-07-11 displaced-attractor rows (si30-class at high ρ, c12/o16-class at low ρ)",
        "RESULTS.md 2026-07-09 Appendix-B rows (the mechanism; commit b8665c0)",
    ],
    scripts=["scripts/step6_label_nse_census.py --witness", "scripts/appendixb_check.py"],
)

## Figure 3 — frozen across dt decades (it is a fixed point, not a slow relaxation)

If the hot labels were merely under-relaxed, the composition would keep
moving as dt grows. On the witness states it does not: the same composition
appears at dt = 1e-6 s and dt = 1e2 s — eight decades. The network has
*converged* to the wrong place. (The dt-freeze was measured on the witness
states, not across the whole census.)

In [ ]:
rows_dt = []
for lab in DT_SHOW:
    d = load_step_frame(NET, lab, columns=[f"final_{n}" for n in names], state_ids=[WITNESS_ID])
    rows_dt.append(d.to_numpy()[0])
Xdt = np.array(rows_dt)  # (9, n_species)

dom = np.argsort(-Xdt[-1])[:5]
fig, ax = plt.subplots(figsize=(10, 4.6))
dts = [float(s) for s in DT_SHOW]
for i in dom:
    ax.loglog(dts, np.maximum(Xdt[:, i], 1e-16), "o-", label=names[i])
if nse_w is not None:
    for i in dom:
        ax.axhline(max(Xn[i], 1e-16), color="#009E73", ls=":", lw=1, alpha=0.6)
    ax.plot([], [], color="#009E73", ls=":", label="true NSE values (dotted)")
ax.set_xlabel("label timestep dt [s]")
ax.set_ylabel("final mass fraction X")
ax.set_title(f"State {WITNESS_ID}: label composition across "
             f"{len(DT_SHOW)} dt values spanning {DT_SHOW[0]} → {DT_SHOW[-1]} s")
if QUICK:
    nbs.quick_banner(fig, "QUICK: 3 of 9 dts — set NB_QUICK=0 for all nine")
ax.legend(fontsize=8, ncol=2)
nbs.caption(
    fig,
    "Flat across eight decades of dt — the displaced composition is an ATTRACTOR, not an "
    "unfinished relaxation. Combined with the three-way witness (stock bbq reproduces the labels "
    "to 0.01–0.03 dex; 24.08.1 relaxes to NSE), this is what makes the attribution dispositive "
    "rather than circumstantial.",
    results=["RESULTS.md 2026-07-11 dt-freeze / three-way witness rows"],
    scripts=["scripts/step6_label_nse_census.py"],
)

## Figure 4 — the consequence: Yₑ itself is wrong

Yₑ is the project's headline quantity. The diagnostic bbq runs at the witness
state's exact conditions — stock r23.05.1 (= the label configuration) vs
MESA 24.08.1 (gh-575 fixed) — disagree on it.

In [ ]:
DIAG = nbs.REPO / "data" / "bbq_reruns"
diag_paths = {
    "stock r23.05.1 (label config)": DIAG / "mesa_80" / "diag_state80_stock" / "output.txt",
    "MESA 24.08.1 (fixed)": DIAG / "mesa_80" / "diag_state80_24081" / "output.txt",
}
have = {k: p for k, p in diag_paths.items() if p.exists()}
if have:
    from gnn_nucleo.data.trajectories import load_trajectory

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    for k, p in have.items():
        tr = load_trajectory(NET, p, source="rerun", logT=float(df["logT"].iloc[pos]),
                             logRho=float(df["logRho"].iloc[pos]))
        ye_t = (tr.X * (Z / A)[None, :]).sum(axis=1)
        c = "#D55E00" if "stock" in k else "#009E73"
        axes[0].semilogx(np.maximum(tr.age, 1e-10), ye_t, lw=1.6, color=c, label=k)
        print(f"{k}: final Yₑ = {ye_t[-1]:.4f}")
        top = np.argsort(-tr.X[-1])[:6]
        axes[1].bar(np.arange(6) + (0.2 if "stock" in k else -0.2),
                    np.maximum(tr.X[-1][top], 1e-14), 0.4, color=c, label=k)
        if "stock" in k:
            axes[1].set_xticks(np.arange(6), [names[i] for i in top], rotation=35, ha="right")
    axes[0].set_xlabel("age [s]")
    axes[0].set_ylabel("Yₑ")
    axes[0].set_title("Yₑ(t) at the witness state's exact conditions")
    axes[0].legend(fontsize=8)
    axes[1].set_yscale("log")
    axes[1].set_ylabel("final X")
    axes[1].set_title("Final composition (stock vs fixed)")
    axes[1].legend(fontsize=7)
    nbs.caption(
        fig,
        "The two MESA versions, same conditions, disagree on the headline quantity: measured 0.4713 "
        "(label/stock) vs 0.4619 (24.08.1) — ~1e-2 in Yₑ, i.e. the size of the entire end-to-end "
        "physics floor (5e-3…1.5e-2 per trajectory), from a rate bug rather than from emulation "
        "error. This is what forces the Step-6 escalation: at T9 ≳ 5, benchmark fidelity to NNN "
        "and physical correctness are DIFFERENT training targets.",
        results=[
            "RESULTS.md 2026-07-11 witness-state Yₑ row (0.4713 label/stock vs 0.4619 on 24.08.1)",
            "root CLAUDE.md end-to-end Yₑ physics floor 5e-3…1.5e-2 per trajectory",
        ],
        scripts=["scripts/step6_label_nse_census.py --witness"],
    )
else:
    print("witness diagnostic runs not found under data/bbq_reruns/mesa_80/diag_state80_*;\n"
          "regenerate with: uv run python scripts/step6_label_nse_census.py --witness\n"
          "Measured result (RESULTS.md 2026-07-11): label/stock Yₑ = 0.4713 vs 24.08.1 Yₑ = 0.4619.")

## Why this is escalated, not just recorded

Two decisions are **blocked on a human** (STEP6_REPORT §6):

1. **Benchmark-vs-physics fork at T9 ≳ 5** — should Phase 1 train on the
   shipped (bug-displaced) labels there (benchmark fidelity to NNN) or on
   corrected physics (24.08.1-class reruns / pf-true Φ labels)? It changes
   supervision, evaluation, and any NNN-comparison claim we make.
2. **External communication** — the pathology implicates the *published* NNN
   training sets, and NNN itself was trained on them. Contacting the
   Grichener et al. authors interacts with our own Paper-1 timing; the
   novelty sweep recommends accelerating.

## What this notebook does NOT show

- The rate-level mechanism (which channels, how wrong): notebook 04.
- That the shipped *trajectories* arrive at the same attractors
  (the "stall"): notebook 07.
- The kill-test consequence — the Guidry maskable set is EMPTY because the
  manifold's own equilibria are displaced: notebook 10.